# Day 1：真正的 Kokoro 美语配音
在 Google Colab 中选择“全部运行”。免费 CPU 即可运行；需要下载公开模型。此备用笔记本尚未在 Colab 实际运行验证。不会登录 GitHub 或修改线上网站。六个 MP3 全部通过编码、时长、能量检查后才生成带按钮的网页。请试听后再部署。

In [ ]:
import subprocess, sys
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "ffmpeg", "espeak-ng"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "kokoro==0.9.4", "transformers==4.49.0", "huggingface-hub==0.29.3", "soundfile==0.14.0"], check=True)


In [ ]:
from pathlib import Path
import json
root = Path("/content/english365-kokoro")
root.mkdir(exist_ok=True)
files = json.loads('{"index.html": "<!doctype html>\\n<html lang=\\"zh-CN\\"><head><meta charset=\\"utf-8\\"><meta name=\\"viewport\\" content=\\"width=device-width,initial-scale=1\\"><meta name=\\"theme-color\\" content=\\"#f6f8fc\\"><title>第1天 Hello｜加拿大生活英语365天</title><style>*{box-sizing:border-box}body{margin:0;background:#f6f8fc;color:#1b2940;font-family:-apple-system,BlinkMacSystemFont,\\"PingFang SC\\",\\"Microsoft YaHei\\",sans-serif}main{max-width:680px;margin:auto;padding:24px 18px 64px}header{margin-bottom:20px}.eyebrow{color:#3566a2;font-size:17px;font-weight:700}h1{font-size:30px;margin:8px 0 6px;line-height:1.3}h2{font-size:24px;margin:28px 0 14px}.sub{color:#627086;font-size:18px;line-height:1.6}.card{background:white;border-radius:20px;padding:24px;margin:14px 0;box-shadow:0 4px 20px rgba(24,45,80,.06)}.word{font-size:clamp(52px,12vw,76px);font-weight:800;color:#1d64aa;line-height:1.2}.pron{font-size:32px;color:#a2482f;margin:14px 0}.meaning{font-size:28px;font-weight:700}.tip{font-size:19px;line-height:1.8;color:#42516b;margin-top:14px}.english{font-size:29px;font-weight:750;line-height:1.45}.chinese{font-size:22px;color:#57677d;margin-top:9px}.notice{background:#fff4d8;border-left:5px solid #d79a30;padding:16px 18px;border-radius:12px;font-size:18px;line-height:1.7}.practice{font-size:21px;line-height:1.7}summary{cursor:pointer;color:#1d64aa;font-weight:700;font-size:21px;padding:8px 0}footer{font-size:15px;color:#6e798c;margin-top:28px;line-height:1.7}</style></head><body><main><header><div class=\\"eyebrow\\">加拿大生活英语365天 · Day 1 / 365</div><h1>今天学会说：你好！</h1><div class=\\"sub\\">为零基础华语长者设计 · 大字 · 简单 · 每天一点点</div></header><section class=\\"card\\"><div class=\\"word\\">Hello!</div><div class=\\"pron\\">哈喽</div><div class=\\"meaning\\">你好！</div><p class=\\"tip\\">发音提示：两个音节，重音在后面的「喽」。中文注音只帮助记忆，不能完全代替英语发音。</p><p class=\\"tip\\">使用场景：见到邻居、朋友或店员时，都可以微笑说 Hello!</p></section><div class=\\"notice\\"><strong>语音制作说明：</strong>这是第1天可阅读试用版。自然美式英语 AI 配音尚未生成和验证，因此暂不放置可能无声的播放按钮。后续会加入单词与例句的正常、慢速音频。</div><h2>三个生活例句</h2><section class=\\"card\\"><div class=\\"english\\">Hello!</div><div class=\\"chinese\\">你好！</div></section><section class=\\"card\\"><div class=\\"english\\">Hello, Mary!</div><div class=\\"chinese\\">玛丽，你好！</div></section><section class=\\"card\\"><div class=\\"english\\">Hello, everyone!</div><div class=\\"chinese\\">大家好！</div></section><h2>今天的小练习</h2><section class=\\"card practice\\"><div>早上在加拿大遇到邻居，您可以说什么？</div><details><summary>点击查看答案</summary><div class=\\"english\\">Hello!</div><div class=\\"chinese\\">你好！</div></details></section><footer>第1天 · Hello · 课程试用版<br>网页上线后，可把网址分享给微信好友。</footer></main></body></html>\\n", "tools/generate_day1.py": "\\"\\"\\"Run real Kokoro inference, validate all six MP3s, then enable page controls.\\"\\"\\"\\nimport argparse\\nimport hashlib\\nimport json\\nimport math\\nfrom pathlib import Path\\nimport subprocess\\nimport tempfile\\n\\nTEXTS = {\'hello\': \'Hello!\', \'hello_mary\': \'Hello, Mary!\', \'hello_everyone\': \'Hello, everyone!\'}\\nSPEEDS = {\'normal\': 1.0, \'slow\': 0.75}\\n\\n\\ndef validate(path):\\n    import numpy as np\\n    info = json.loads(subprocess.check_output([\\n        \'ffprobe\', \'-v\', \'error\', \'-show_streams\', \'-show_format\', \'-of\', \'json\', str(path)]))\\n    stream = next(s for s in info[\'streams\'] if s[\'codec_type\'] == \'audio\')\\n    raw = subprocess.check_output([\'ffmpeg\', \'-v\', \'error\', \'-i\', str(path),\\n                                   \'-f\', \'f32le\', \'-ac\', \'1\', \'-ar\', \'24000\', \'-\'])\\n    samples = np.frombuffer(raw, dtype=\'<f4\')\\n    rms = float(np.sqrt(np.mean(samples ** 2)))\\n    peak = float(np.max(np.abs(samples))) if len(samples) else 0\\n    duration = float(info[\'format\'][\'duration\'])\\n    if not (stream[\'codec_name\'] == \'mp3\' and path.stat().st_size > 1000\\n            and int(stream[\'sample_rate\']) == 24000 and 0.2 < duration < 20 and np.isfinite(samples).all()\\n            and math.isfinite(rms) and rms > 0.001 and peak > 0.01):\\n        raise ValueError(f\'Invalid or silent audio: {path}\')\\n    return dict(bytes=path.stat().st_size, duration_seconds=duration,\\n                codec=stream[\'codec_name\'], sample_rate=int(stream[\'sample_rate\']),\\n                rms=rms, peak=peak, sha256=hashlib.sha256(path.read_bytes()).hexdigest(),\\n                decoded_sha256=hashlib.sha256(raw).hexdigest())\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser()\\n    parser.add_argument(\'--root\', type=Path, default=Path(__file__).resolve().parents[1])\\n    args = parser.parse_args()\\n    root = args.root.resolve()\\n    original = (root / \'index.html\').read_text(encoding=\'utf-8\')\\n    if \'id=\\"day1-player\\"\' in original:\\n        raise SystemExit(\'Page already has audio controls; inspect before regenerating.\')\\n    # Check the expected page before downloading a model or writing any output.\\n    for text in TEXTS.values():\\n        if f\'<div class=\\"english\\">{text}</div>\' not in original:\\n            raise SystemExit(f\'Expected lesson text missing: {text}\')\\n    import numpy as np\\n    import soundfile as sf\\n    import torch\\n    from kokoro import KPipeline\\n    torch.set_num_threads(min(4, torch.get_num_threads()))\\n    pipeline = KPipeline(lang_code=\'a\', repo_id=\'hexgrad/Kokoro-82M\', device=\'cpu\')\\n    report = {\'model\': \'hexgrad/Kokoro-82M\', \'voice\': \'af_heart\',\\n              \'speeds\': SPEEDS, \'method\': \'Kokoro inference; no time stretching\',\\n              \'listening_verified\': False, \'files\': {}}\\n    # Stage outside the repository: no buttons or audio are published on failure.\\n    with tempfile.TemporaryDirectory(prefix=\'english365-kokoro-\') as tmp:\\n        staging = Path(tmp)\\n        for key, text in TEXTS.items():\\n            for label, speed in SPEEDS.items():\\n                chunks = [audio.detach().cpu().numpy() for _, _, audio in\\n                          pipeline(text, voice=\'af_heart\', speed=speed) if audio is not None]\\n                if not chunks:\\n                    raise RuntimeError(f\'Kokoro returned no audio: {text}\')\\n                waveform = np.concatenate(chunks)\\n                wav = staging / f\'{key}_{label}.wav\'\\n                mp3 = staging / f\'{key}_{label}.mp3\'\\n                sf.write(wav, waveform, 24000, subtype=\'PCM_16\')\\n                subprocess.run([\'ffmpeg\', \'-v\', \'error\', \'-nostdin\', \'-y\', \'-i\', str(wav),\\n                                \'-codec:a\', \'libmp3lame\', \'-b:a\', \'128k\', str(mp3)], check=True)\\n                report[\'files\'][mp3.name] = dict(text=text, speed=speed, **validate(mp3))\\n                print(mp3.name, report[\'files\'][mp3.name], flush=True)\\n        for field in (\'sha256\', \'decoded_sha256\'):\\n            if len({item[field] for item in report[\'files\'].values()}) != 6:\\n                raise RuntimeError(f\'Duplicate audio detected: {field}\')\\n        for key in TEXTS:\\n            if report[\'files\'][key+\'_slow.mp3\'][\'duration_seconds\'] <= report[\'files\'][key+\'_normal.mp3\'][\'duration_seconds\']:\\n                raise RuntimeError(f\'Slow audio is not longer: {key}\')\\n        from upgrade_page import upgrade\\n        updated = upgrade(original)\\n        (root / \'audio\').mkdir(exist_ok=True)\\n        for mp3 in staging.glob(\'*.mp3\'):\\n            dest = root / \'audio\' / mp3.name\\n            if dest.exists():\\n                raise FileExistsError(f\'Refusing to overwrite {dest}\')\\n        for mp3 in staging.glob(\'*.mp3\'):\\n            (root / \'audio\' / mp3.name).write_bytes(mp3.read_bytes())\\n        (root / \'audio\' / \'validation.json\').write_text(json.dumps(report, indent=2), encoding=\'utf-8\')\\n        (root / \'index.html\').write_text(updated, encoding=\'utf-8\')\\n    print(\'Six MP3s validated and HTML upgraded. Listening and browser/device checks remain required.\')\\n\\n\\nif __name__ == \'__main__\':\\n    main()\\n", "tools/upgrade_page.py": "\\"\\"\\"Called only after all audio files pass validation; preserves lesson content.\\"\\"\\"\\nimport re\\n\\nCSS = \'\'\'\\n.audio-controls{display:flex;flex-wrap:wrap;gap:12px;margin-top:18px}\\n.audio-controls button{min-height:56px;flex:1 1 160px;padding:14px 18px;border:2px solid #1d64aa;border-radius:14px;background:#1d64aa;color:white;font-weight:700;font-size:20px;line-height:1.4;font-family:inherit;cursor:pointer;touch-action:manipulation}\\n.audio-controls button:focus-visible{outline:3px solid #a2482f;outline-offset:3px}\\n.audio-controls button[aria-pressed=\\"true\\"]{background:#17436e}\\n#audio-status{font-size:19px;line-height:1.7;overflow-wrap:anywhere}\\n\'\'\'\\nJS = \'\'\'\\n(function () {\\n  \'use strict\';\\n  var player = document.getElementById(\'day1-player\');\\n  var status = document.getElementById(\'audio-status\');\\n  var buttons = document.querySelectorAll(\'[data-audio]\');\\n  var current = null;\\n  var request = 0;\\n  function reset() {\\n    buttons.forEach(function (button) { button.setAttribute(\'aria-pressed\', \'false\'); });\\n  }\\n  function failed(id) {\\n    if (id !== request) return;\\n    reset();\\n    status.textContent = \'播放失败，请检查网络后再点一次。若仍无声音，请在手机浏览器中打开本页，并检查手机音量。\';\\n  }\\n  buttons.forEach(function (button) {\\n    button.addEventListener(\'click\', function () {\\n      var id = ++request;\\n      player.pause();\\n      reset();\\n      current = button;\\n      player.src = button.getAttribute(\'data-audio\');\\n      status.textContent = \'正在加载：\' + button.getAttribute(\'aria-label\');\\n      button.setAttribute(\'aria-pressed\', \'true\');\\n      // Call play synchronously within the user gesture for iOS/WebView support.\\n      try {\\n        var result = player.play();\\n        if (result && typeof result.catch === \'function\') result.catch(function () { failed(id); });\\n      } catch (error) { failed(id); }\\n    });\\n  });\\n  player.addEventListener(\'playing\', function () {\\n    if (current) status.textContent = \'正在播放：\' + current.getAttribute(\'aria-label\');\\n  });\\n  player.addEventListener(\'ended\', function () { reset(); status.textContent = \'播放完毕，可以再听一次。\'; });\\n  player.addEventListener(\'error\', function () { if (player.error) failed(request); });\\n}());\\n\'\'\'\\n\\n\\ndef controls(key, text):\\n    return \'<div class=\\"audio-controls\\">\' + \'\'.join(\\n        f\'<button type=\\"button\\" data-audio=\\"audio/{key}_{speed}.mp3\\" aria-label=\\"{text}，{label}\\" aria-pressed=\\"false\\">{label}</button>\'\\n        for speed, label in [(\'normal\', \'正常发音\'), (\'slow\', \'慢速发音\')]) + \'</div>\'\\n\\n\\ndef upgrade(html):\\n    if \'id=\\"day1-player\\"\' in html:\\n        raise ValueError(\'Already upgraded\')\\n    html = html.replace(\'</style>\', CSS + \'</style>\', 1)\\n    word = \'<div class=\\"meaning\\">你好！</div>\'\\n    if html.count(word) != 1:\\n        raise ValueError(\'Word card does not match expected content\')\\n    html = html.replace(word, word + controls(\'hello\', \'Hello!\'), 1)\\n    for key, text in [(\'hello\', \'Hello!\'), (\'hello_mary\', \'Hello, Mary!\'), (\'hello_everyone\', \'Hello, everyone!\')]:\\n        target = f\'<section class=\\"card\\"><div class=\\"english\\">{text}</div>\'\\n        if html.count(target) != 1:\\n            raise ValueError(f\'Example card does not match: {text}\')\\n        html = html.replace(target, target + controls(key, text), 1)\\n    html, count = re.subn(r\'<div class=\\"notice\\">.*?</div>\',\\n        \'<div class=\\"notice\\"><strong>听一听：</strong>点击“正常发音”或“慢速发音”，跟着美语女声练习。音频由 Kokoro-82M 的 af_heart 声音生成。<div id=\\"audio-status\\" role=\\"status\\" aria-live=\\"polite\\">点击按钮即可播放，请先调好手机音量。</div></div>\', html, count=1, flags=re.S)\\n    if count != 1:\\n        raise ValueError(\'Expected audio notice missing\')\\n    return html.replace(\'</body>\', \'<audio id=\\"day1-player\\" preload=\\"none\\"></audio><script>\' + JS + \'</script></body>\', 1)\\n"}')
for name, content in files.items():
    target = root / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding="utf-8")
subprocess.run([sys.executable, str(root / "tools/generate_day1.py"), "--root", str(root)], check=True)


In [ ]:
from IPython.display import Audio, display
for path in sorted((root / "audio").glob("*.mp3")):
    print(path.name)
    display(Audio(filename=str(path)))
print("请逐个试听：确认内容、正常/慢速、音量和自然程度。机器检查不能代替试听。")


In [ ]:
import zipfile
from google.colab import files
archive = Path("/content/english365-day1-kokoro.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as out:
    out.write(root / "index.html", "index.html")
    for path in (root / "audio").iterdir():
        out.write(path, "audio/" + path.name)
files.download(str(archive))


解压后的 index.html 与 audio/ 必须放在同一层。试听通过后，可回到 Codex 继续提交和部署；本笔记本不会自动部署。实机测试：iPhone Safari、Android Chrome、微信各打开网站，依次点击四组正常/慢速按钮，检查声音、音量、中文提示和竖屏布局；快速切换按钮不得叠音；断网点击未缓存音频应显示清楚的失败提示。